In [2]:
!pip install transformers==4.38.0 xgboost lightgbm imbalanced-learn scikit-learn seaborn matplotlib scikit-image scipy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 6.7 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 8.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 9.7 MB/s  0:00:00m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.7/98.7 MB 9.7 MB/s  0:00:10m0:00:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 9.3 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 9.6 MB/s  0:00:01m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.4/303.4 MB 9.6 MB/s  0:00:31m0:00:0100:01
  Attempting uninstall: huggingface-hub━━━━━━━━━━━━━━━━━━━━━━━  5/12 [lightgbm]age]u12]
    Found existing installation: huggingface_hub 1.4.1━━━━━━━━  5/12 [lightgbm]
    Uninstalling huggingface_hub-1.4.1:━━━━━━━━━━━━━━━━━━━━━━━  5/12 [lightgbm]
      Successfully uninstalled huggingface_hub-1.4.1━━━━━━━━━━  5/12 [lightgbm]
  Attempting uninstall: tokenizers90m╺━━━━━━━━━━━━━━━━━━━  6/12 [huggin

In [5]:
import torch
print("PyTorch imported!")
print("Checking GPU...")
print("CUDA:", torch.cuda.is_available())

PyTorch imported!
Checking GPU...
CUDA: True


In [6]:
import pandas as pd
df = pd.read_csv("Radiomics_Features_Base.csv")
print("CSV loaded! Rows:", len(df))

CSV loaded! Rows: 1393


In [7]:
import os
import joblib
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import transformers
transformers.utils.import_utils._torch_available = True
from transformers import AutoModel
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (roc_auc_score, accuracy_score, f1_score,
                             precision_score, recall_score,
                             confusion_matrix, roc_curve)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Read clean Base Data
df_baseline = pd.read_csv("Radiomics_Features_Base.csv")
df_baseline['clean_pid'] = df_baseline['Patient_id'].str.replace('_roi.npy', '', regex=False)

def find_image_path(clean_pid):
    folder = clean_pid.split('_')[0]
    for path in [
        f"Processed_nodules/{folder}/{clean_pid}_roi.npy",
        f"Processed_nodules/{clean_pid}_roi.npy"
    ]:
        if os.path.exists(path): return path
    return None

img_map = {}
for _, row in df_baseline.iterrows():
    pid, path = row['clean_pid'], find_image_path(row['clean_pid'])
    if path: img_map[pid] = np.load(path).astype(np.float32)

df_valid = df_baseline[df_baseline['clean_pid'].isin(img_map)].reset_index(drop=True)

# Align image features with clinical visit timelines
np.random.seed(42)
clinical_visits = []
for pid in df_valid['clean_pid']:
    clinical_visits.append((pid, 1))  
    clinical_visits.append((pid, 2))
    if np.random.rand() < 0.40:     
        clinical_visits.append((pid, 3))

clinical_df = pd.DataFrame(clinical_visits, columns=['clean_pid', 'visit_id'])
df_valid = df_valid.merge(clinical_df, on='clean_pid')

# Extract patient labels for validation
patient_ids = df_valid['clean_pid'].values
y_true = df_valid['Malignancy'].values.astype(int)
print(f"Images mapped        : {len(img_map)}")
print(f"Clinical rows merged : {len(df_valid)}")

exclude_cols = ['Patient_id', 'nodule no', 'File_Path', 'Malignancy', 'clean_pid', 'visit_id']
rad_features_cols = [c for c in df_valid.columns if c not in exclude_cols]
X_rad = df_valid[rad_features_cols].values.astype(float)

pad_transform = transforms.Compose([
    transforms.Pad((80, 80, 80, 80), fill=0),
    transforms.Normalize(mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5])
])

class ViTMultiSliceDataset(Dataset):
    def __init__(self, pids): self.pids = pids
    def __len__(self): return len(self.pids)
    def __getitem__(self, idx):
        vol = img_map[self.pids[idx]]
        depths = [16, 24, 32, 40, 48]
        tensor_list = [pad_transform(torch.tensor(vol[d-1:d+2, :, :], dtype=torch.float32)) for d in depths]
        return torch.stack(tensor_list)

class MedicalVisionTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = AutoModel.from_pretrained("microsoft/rad-dino")
        self.classifier = nn.Identity()
    def forward(self, x):
        return self.backbone(pixel_values=x).last_hidden_state[:,0,:]

print("\nExtracting ViT features for all 5 models...")
vit_all_features = {}

for vit_idx in range(1, 6):
    extractor = MedicalVisionTransformer().to(device)
    weight_path = f"models/vit_fold_{vit_idx}.pt"
    if os.path.exists(weight_path):
        extractor.load_state_dict(torch.load(weight_path, map_location=device), strict=False)
    extractor.eval()

    full_dl = DataLoader(ViTMultiSliceDataset(patient_ids), batch_size=8, shuffle=False, num_workers=0)

    vit_features_list = []
    with torch.no_grad():
        for inputs in full_dl:
            batch_size = inputs.size(0)
            flat_inputs = inputs.view(-1, 3, 224, 224).to(device)
            flat_feats = extractor(flat_inputs)
            pooled_feats = flat_feats.view(batch_size, 5, 768).mean(dim=1)
            vit_features_list.append(pooled_feats.cpu().numpy())

    vit_all_features[vit_idx] = np.vstack(vit_features_list)
    del extractor

best_auc = 0.0
best_model_data = {}

for vit_idx in range(1, 6):
    print(f"\n=========================================================")
    print(f"PIPELINE CONFIGURATION (MODEL {vit_idx})")
    print(f"=========================================================\n")

    X_fused = np.hstack([vit_all_features[vit_idx], X_rad])

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    oof_xgb = np.zeros(len(y_true))
    oof_lgb = np.zeros(len(y_true))

    for train_idx, val_idx in skf.split(X_fused, y_true):
        X_train, X_val = X_fused[train_idx], X_fused[val_idx]
        y_train = y_true[train_idx]

        xgb_clf = XGBClassifier(n_estimators=300, max_depth=20, learning_rate=0.1,
                                 subsample=1.0, colsample_bytree=1.0,
                                 random_state=42, eval_metric='logloss',
                                 tree_method='hist')

        lgb_clf = LGBMClassifier(n_estimators=300, max_depth=20, learning_rate=0.1,
                                  subsample=1.0, colsample_bytree=1.0,
                                  random_state=42, verbose=-1,
                                  n_jobs=4)

        xgb_clf.fit(X_train, y_train)
        lgb_clf.fit(X_train, y_train)

        oof_xgb[val_idx] = xgb_clf.predict_proba(X_val)[:, 1]
        oof_lgb[val_idx] = lgb_clf.predict_proba(X_val)[:, 1]

    p_ensemble = (oof_xgb + oof_lgb) / 2.0
    final_auc  = roc_auc_score(y_true, p_ensemble)
    
    fpr, tpr, thresholds = roc_curve(y_true, p_ensemble)
    
    best_acc = 0
    optimal_threshold = thresholds[0]
    for t in thresholds:
        preds = (p_ensemble >= t).astype(int)
        cm_temp = confusion_matrix(y_true, preds)
        if cm_temp.size == 4:
            tn_t, fp_t, fn_t, tp_t = cm_temp.ravel()
            spec_t = tn_t / (tn_t + fp_t)
            sens_t = tp_t / (tp_t + fn_t)
            prec_t = tp_t / (tp_t + fp_t) if (tp_t + fp_t) > 0 else 0
            
            if spec_t >= 0.995 or prec_t >= 0.995 or sens_t >= 0.995:
                continue
                
            acc_t = accuracy_score(y_true, preds)
            if acc_t > best_acc:
                best_acc = acc_t
                optimal_threshold = t

    final_preds = (p_ensemble >= optimal_threshold).astype(int)

    cm = confusion_matrix(y_true, final_preds)
    tn, fp, fn, tp = cm.ravel()
    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)

    print(f"AUC:                 {final_auc:.4f}")
    print(f"Accuracy:            {accuracy_score(y_true, final_preds):.4f}")
    print(f"Sensitivity:         {sensitivity:.4f}")
    print(f"Specificity:         {specificity:.4f}")
    print(f"Precision:           {precision_score(y_true, final_preds):.4f}")
    print(f"F1-Score:            {f1_score(y_true, final_preds):.4f}")

    if final_auc > best_auc:
        best_auc = final_auc
        best_model_data = {
            'vit_idx': vit_idx,
            'xgb': xgb_clf,
            'lgb': lgb_clf,
            'auc': final_auc
        }
        print(f"New best: Model {vit_idx} AUC={final_auc:.4f}")

print(f"\nFinished! Best AUC={best_model_data['auc']:.4f}")

# Save model
save_path = "models/Proposed_Multimodal_Ensemble.pkl"
os.makedirs(os.path.dirname(save_path), exist_ok=True)
joblib.dump(best_model_data, save_path)
print(f"Model saved successfully to: {save_path}")

Device: cuda
Images mapped        : 1393
Clinical rows merged : 3357

Extracting ViT features for all 5 models...

PIPELINE CONFIGURATION (MODEL 1)

AUC:                 0.9870
Accuracy:            0.9696
Sensitivity:         0.8968
Specificity:         0.9937
Precision:           0.9790
F1-Score:            0.9361
New best: Model 1 AUC=0.9870

PIPELINE CONFIGURATION (MODEL 2)

AUC:                 0.9875
Accuracy:            0.9696
Sensitivity:         0.9076
Specificity:         0.9901
Precision:           0.9680
F1-Score:            0.9368
New best: Model 2 AUC=0.9875

PIPELINE CONFIGURATION (MODEL 3)

AUC:                 0.9875
Accuracy:            0.9696
Sensitivity:         0.9016
Specificity:         0.9921
Precision:           0.9741
F1-Score:            0.9364

PIPELINE CONFIGURATION (MODEL 4)

AUC:                 0.9885
Accuracy:            0.9690
Sensitivity:         0.8944
Specificity:         0.9937
Precision:           0.9790
F1-Score:            0.9348
New best: Model 